# Modeling - Interpolation Baseline

A deliberately naive anchor model. It predicts target-zone TVT by linearly interpolating or forward-filling the known `TVT_input` series within each well.

## 1. Setup

Load project paths, data, metric helpers, and the train wells. (Imports and constants.)

In [ ]:

import sys
import time
from datetime import date
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    for parent in ROOT.parents:
        if (parent / "src").is_dir():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

from src.evaluation.metrics import metrics_summary, rmse
from src.evaluation.validation import group_well_folds, known_zone_mask, target_zone_mask
from src.io.loaders import discover_wells, load_all_wells
from src.modeling.baseline import InterpolationBaseline
from src.modeling.linear import LinearRegressor, Standardizer
from src.utils.paths import get_artifacts_dir, get_data_dir
from src.utils.run_bundle import RunBundle
from src.utils.seeds import set_global_seeds

set_global_seeds(42)
RUN_DATE = date.today().isoformat()
N_SPLITS = 3
MAX_TRAIN_ROWS_PER_FOLD = 25_000

train_dir = get_data_dir("train")
figures_dir = get_artifacts_dir("figures")
tables_dir = get_artifacts_dir("tables")
models_dir = get_artifacts_dir("models")
figures_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

wells = discover_wells(train_dir)
well_bundle = load_all_wells(train_dir, wells=wells)
folds = group_well_folds(wells, n_splits=N_SPLITS)

print(f"train wells: {len(wells)}")
print(f"folds: {len(folds)}")


## 2. Shared Helpers

Small notebook helpers for artifact writing and simple residual visualizations. (Artifact utilities.)

In [ ]:

def _safe_gr(series):
    gr = series.clip(lower=0, upper=300)
    return gr.fillna(gr.rolling(21, center=True, min_periods=1).median()).ffill().bfill()


def make_feature_frame(df):
    out = pd.DataFrame(index=df.index)
    out["row_idx"] = df["row_idx"].astype(float)
    out["MD"] = df["MD"].astype(float)
    out["X"] = df["X"].astype(float)
    out["Y"] = df["Y"].astype(float)
    out["Z"] = df["Z"].astype(float)
    out["GR"] = _safe_gr(df["GR"].astype(float))
    out["TVT_input_ffill"] = df["TVT_input"].ffill()
    out["known_zone"] = df["TVT_input"].notna().astype(float)
    out["md_from_start"] = out["MD"] - out["MD"].min()
    out["md_norm"] = out["md_from_start"] / (out["md_from_start"].max() + 1e-9)
    out["dX"] = out["X"].diff().fillna(0.0)
    out["dY"] = out["Y"].diff().fillna(0.0)
    out["dZ"] = out["Z"].diff().fillna(0.0)
    out["horizontal_step"] = np.sqrt(out["dX"] ** 2 + out["dY"] ** 2)
    out["horizontal_dist"] = out["horizontal_step"].cumsum()
    return out


def train_valid_arrays(well_bundle, train_wells, valid_wells):
    train_frames = []
    valid_frames = []
    valid_targets = []
    valid_meta = []

    for well in train_wells:
        df = well_bundle[well]["horizontal"]
        mask = known_zone_mask(df)
        feat = make_feature_frame(df).loc[mask]
        feat["target"] = df.loc[mask, "TVT"].to_numpy()
        train_frames.append(feat)

    for well in valid_wells:
        df = well_bundle[well]["horizontal"]
        mask = target_zone_mask(df)
        feat = make_feature_frame(df).loc[mask]
        valid_frames.append(feat)
        valid_targets.append(df.loc[mask, "TVT"].to_numpy())
        valid_meta.extend([well] * int(mask.sum()))

    train_df = pd.concat(train_frames, axis=0)
    valid_df = pd.concat(valid_frames, axis=0)
    y_train = train_df.pop("target").to_numpy(dtype=float)
    y_valid = np.concatenate(valid_targets).astype(float)

    medians = train_df.median(numeric_only=True)
    X_train = train_df.fillna(medians).to_numpy(dtype=float)
    X_valid = valid_df.fillna(medians).to_numpy(dtype=float)
    return X_train, y_train, X_valid, y_valid, valid_meta, list(train_df.columns)


def sample_training_rows(X, y, max_rows):
    if len(y) <= max_rows:
        return X, y
    idx = np.linspace(0, len(y) - 1, max_rows).astype(int)
    return X[idx], y[idx]


def append_model_comparison(row):
    path = tables_dir / "model_comparison.csv"
    if path.exists():
        table = pd.read_csv(path)
        table = pd.concat([table, pd.DataFrame([row])], ignore_index=True)
    else:
        table = pd.DataFrame([row])
    table = table.drop_duplicates(["model_family"], keep="last")
    table.to_csv(path, index=False)
    return path


def save_metric_bars(metrics_df, path, title):
    width, height = 920, 360
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    draw.text((24, 18), title, fill="black")
    values = metrics_df["rmse"].to_numpy(dtype=float)
    labels = metrics_df["model"].astype(str).tolist()
    max_value = max(float(values.max()), 1.0)
    left, top, bar_h = 190, 70, 34
    for i, (label, value) in enumerate(zip(labels, values)):
        y = top + i * 52
        draw.text((24, y + 8), label[:22], fill="black")
        bar_w = int((width - left - 80) * value / max_value)
        draw.rectangle((left, y, left + bar_w, y + bar_h), fill="#4c78a8")
        draw.text((left + bar_w + 8, y + 8), f"{value:.4f}", fill="black")
    image.save(path)
    return path


def save_residual_strip(y_true, y_pred, path, title):
    width, height = 920, 360
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    draw.text((24, 18), title, fill="black")
    residual = np.asarray(y_pred) - np.asarray(y_true)
    if residual.size > 1400:
        idx = np.linspace(0, residual.size - 1, 1400).astype(int)
        residual = residual[idx]
    lo, hi = np.percentile(residual, [1, 99])
    bound = max(abs(float(lo)), abs(float(hi)), 1.0)
    mid = height // 2
    draw.line((40, mid, width - 30, mid), fill="#444444")
    for i, value in enumerate(residual):
        x = 40 + int(i * (width - 80) / max(len(residual) - 1, 1))
        y = mid - int((float(value) / bound) * 120)
        color = "#d95f02" if value > 0 else "#1b9e77"
        draw.line((x, mid, x, y), fill=color)
    draw.text((24, height - 34), f"sampled residuals; clipped at +/-{bound:.2f}", fill="black")
    image.save(path)
    return path


## 3. Grouped Masked-Zone Evaluation

Score only rows where `TVT_input` is missing and true `TVT` exists. (Competition-like validation.)

In [ ]:

start = time.time()
model = InterpolationBaseline()
fold_rows = []
all_true = []
all_pred = []

for fold_idx, (_, valid_wells) in enumerate(folds, start=1):
    fold_true = []
    fold_pred = []
    for well in valid_wells:
        df = well_bundle[well]["horizontal"]
        mask = target_zone_mask(df)
        pred = model.predict_well(df)[mask.to_numpy()]
        true = df.loc[mask, "TVT"].to_numpy(dtype=float)
        fold_true.append(true)
        fold_pred.append(pred)
    y_true = np.concatenate(fold_true)
    y_pred = np.concatenate(fold_pred)
    summary = metrics_summary(y_true, y_pred)
    summary.update({"fold": fold_idx, "n_wells": len(valid_wells), "n_rows": len(y_true)})
    fold_rows.append(summary)
    all_true.append(y_true)
    all_pred.append(y_pred)

metrics_df = pd.DataFrame(fold_rows)
overall_true = np.concatenate(all_true)
overall_pred = np.concatenate(all_pred)
overall = metrics_summary(overall_true, overall_pred)
train_time_s = time.time() - start

metrics_df


## 4. Artifacts And Run Bundle

Export fold metrics, a residual plot, the run bundle, and the master model-comparison row. (Reportable outputs.)

In [ ]:

config = {
    "model": "InterpolationBaseline",
    "validation": "3-fold grouped by well; score TVT_input-missing rows",
    "n_splits": N_SPLITS,
}
with RunBundle("interpolation_baseline", config) as run:
    table_path = tables_dir / f"model__interpolation_baseline_metrics__{run.run_id}.csv"
    figure_path = figures_dir / f"model__interpolation_baseline_residuals__{run.run_id}.png"
    metrics_df.to_csv(table_path, index=False)
    save_residual_strip(overall_true, overall_pred, figure_path, "Interpolation baseline residuals")
    run.add_input(str(train_dir))
    run.add_output(str(table_path))
    run.add_output(str(figure_path))
    run.add_metric("cv_rmse_mean", float(metrics_df["rmse"].mean()))
    run.add_metric("cv_rmse_std", float(metrics_df["rmse"].std(ddof=0)))
    run.add_metric("overall_rmse", float(overall["rmse"]))
    run.set_notes(
        "Naive interpolation baseline. Predicts evaluation-zone TVT by linear interpolation/forward fill of TVT_input."
    )

comparison_path = append_model_comparison(
    {
        "run_id": run.run_id,
        "model_family": "baseline",
        "model_name": "interpolation",
        "cv_rmse_mean": float(metrics_df["rmse"].mean()),
        "cv_rmse_std": float(metrics_df["rmse"].std(ddof=0)),
        "train_time_s": float(train_time_s),
        "n_features": 1,
        "notes": "Stateless interpolation/ffill baseline",
    }
)

print("run_id:", run.run_id)
print("overall_rmse:", overall["rmse"])
print("metrics:", table_path)
print("figure:", figure_path)
print("comparison:", comparison_path)
